In [0]:
import math

CATALOG   = "e-commerce"
SILVER_DB = "silver"
GOLD_DB   = "gold"

In [0]:
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

spark.table(f"{SILVER_DB}.fact_sales").createOrReplaceTempView("fact_sales")
spark.table(f"{SILVER_DB}.dim_customers").createOrReplaceTempView("dim_customers")
spark.table(f"{SILVER_DB}.dim_calendar").createOrReplaceTempView("dim_calendar")
print("Gold DB ready | Silver views registered")

In [0]:
df_customer_revenue = spark.sql("""
    SELECT
        cu.customer_sk,
        cu.customer_id,
        cu.age,
        cu.age_band,
        cu.gender,
        cu.loyalty_label,
        cu.join_date,

        COUNT(f.order_id)                               AS total_orders,
        SUM(f.quantity)                                 AS total_units_bought,
        ROUND(SUM(f.revenue),  2)                       AS total_revenue,
        ROUND(SUM(f.profit),   2)                       AS total_profit,
        ROUND(AVG(f.profit_margin)*100, 2)              AS avg_margin_pct,
        ROUND(SUM(f.revenue) / COUNT(f.order_id), 2)   AS avg_order_value,
        ROUND(SUM(f.discount_amount), 2)                AS total_discount_received,

        CASE
            WHEN PERCENT_RANK() OVER (ORDER BY SUM(f.revenue) DESC) <= 0.10
                THEN 'Platinum - Top 10%'
            WHEN PERCENT_RANK() OVER (ORDER BY SUM(f.revenue) DESC) <= 0.25
                THEN 'Gold - Top 25%'
            WHEN PERCENT_RANK() OVER (ORDER BY SUM(f.revenue) DESC) <= 0.50
                THEN 'Silver - Top 50%'
            ELSE 'Bronze'
        END AS customer_tier,

        RANK() OVER (ORDER BY SUM(f.revenue) DESC)      AS revenue_rank

    FROM fact_sales    f
    JOIN dim_customers cu ON f.customer_sk = cu.customer_sk
    WHERE f.is_valid = 1
    GROUP BY
        cu.customer_sk, cu.customer_id, cu.age,
        cu.age_band, cu.gender, cu.loyalty_label, cu.join_date
    ORDER BY total_revenue DESC
""")

print("TOP 20 CUSTOMERS BY REVENUE")
df_customer_revenue.show(20, truncate=False)

row_count = df_customer_revenue.count()
num_partitions = max(2, math.ceil(row_count / 10000))

df_customer_revenue.repartition(num_partitions).write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.customer_revenue_ranking")

print(f"gold.customer_revenue_ranking written - {row_count} customers ({num_partitions} partitions)")

In [0]:
df_age_band = spark.sql("""
    SELECT
        cu.age_band,

        COUNT(DISTINCT cu.customer_sk)                  AS unique_customers,
        COUNT(f.order_id)                               AS total_orders,
        SUM(f.quantity)                                 AS total_units,
        ROUND(SUM(f.revenue),  2)                       AS total_revenue,
        ROUND(SUM(f.profit),   2)                       AS total_profit,
        ROUND(AVG(f.profit_margin)*100, 2)              AS avg_margin_pct,
        ROUND(SUM(f.revenue) / COUNT(DISTINCT cu.customer_sk), 2) AS revenue_per_customer,
        ROUND(SUM(f.revenue) / COUNT(f.order_id), 2)   AS aov,

        ROUND(
            SUM(f.revenue) * 100.0
            / SUM(SUM(f.revenue)) OVER (), 2
        )                                               AS revenue_share_pct,

        RANK() OVER (ORDER BY SUM(f.revenue) DESC)      AS revenue_rank

    FROM fact_sales    f
    JOIN dim_customers cu ON f.customer_sk = cu.customer_sk
    WHERE f.is_valid = 1
    GROUP BY cu.age_band
    ORDER BY revenue_rank
""")

print("REVENUE BY AGE BAND")
df_age_band.show(truncate=False)

row_count = df_age_band.count()
num_partitions = max(2, math.ceil(row_count / 10000))

df_age_band.repartition(num_partitions).write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.revenue_by_age_band")

print(f"gold.revenue_by_age_band written - {row_count} rows")

In [0]:
df_loyalty = spark.sql("""
    SELECT
        cu.loyalty_label,
        cu.loyalty_member,

        COUNT(DISTINCT cu.customer_sk)                  AS unique_customers,
        COUNT(f.order_id)                               AS total_orders,
        SUM(f.quantity)                                 AS total_units,
        ROUND(SUM(f.revenue),    2)                     AS total_revenue,
        ROUND(SUM(f.profit),     2)                     AS total_profit,
        ROUND(AVG(f.profit_margin)*100, 2)              AS avg_margin_pct,

        ROUND(SUM(f.revenue)  / COUNT(DISTINCT cu.customer_sk), 2) AS revenue_per_customer,
        ROUND(COUNT(f.order_id) / COUNT(DISTINCT cu.customer_sk), 2) AS orders_per_customer,
        ROUND(SUM(f.revenue) / COUNT(f.order_id), 2)  AS aov,

        ROUND(
            SUM(f.revenue) * 100.0
            / SUM(SUM(f.revenue)) OVER (), 2
        )                                               AS revenue_share_pct

    FROM fact_sales    f
    JOIN dim_customers cu ON f.customer_sk = cu.customer_sk
    WHERE f.is_valid = 1
    GROUP BY cu.loyalty_label, cu.loyalty_member
    ORDER BY cu.loyalty_member DESC
""")

print("LOYALTY vs NON-LOYALTY")
df_loyalty.show(truncate=False)

print("BY AGE BAND x LOYALTY")
spark.sql("""
    SELECT
        cu.age_band,
        cu.loyalty_label,
        COUNT(DISTINCT cu.customer_sk)  AS customers,
        ROUND(SUM(f.revenue), 2)        AS total_revenue,
        ROUND(SUM(f.revenue) / COUNT(DISTINCT cu.customer_sk), 2) AS rev_per_customer
    FROM fact_sales    f
    JOIN dim_customers cu ON f.customer_sk = cu.customer_sk
    WHERE f.is_valid = 1
    GROUP BY cu.age_band, cu.loyalty_label
    ORDER BY cu.age_band, cu.loyalty_label
""").show(20, truncate=False)

row_count = df_loyalty.count()
num_partitions = max(2, math.ceil(row_count / 10000))

df_loyalty.repartition(num_partitions).write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.loyalty_vs_nonloyalty")

print(f"gold.loyalty_vs_nonloyalty written - {row_count} rows")

In [0]:
df_spend = spark.sql("""
    WITH customer_totals AS (
        SELECT
            f.customer_sk,
            COUNT(f.order_id)           AS orders,
            SUM(f.revenue)              AS total_spend,
            SUM(f.quantity)             AS total_units
        FROM fact_sales f
        WHERE f.is_valid = 1
        GROUP BY f.customer_sk
    )
    SELECT
        ROUND(AVG(total_spend), 2)      AS avg_spend_per_customer,
        ROUND(AVG(orders), 2)           AS avg_orders_per_customer,
        ROUND(AVG(total_units), 2)      AS avg_units_per_customer,

        COUNT(CASE WHEN total_spend < 50   THEN 1 END) AS spend_under_50,
        COUNT(CASE WHEN total_spend BETWEEN 50 AND 100  THEN 1 END) AS spend_50_100,
        COUNT(CASE WHEN total_spend BETWEEN 100 AND 200 THEN 1 END) AS spend_100_200,
        COUNT(CASE WHEN total_spend > 200  THEN 1 END) AS spend_over_200,

        COUNT(*)                        AS total_customers,
        ROUND(MIN(total_spend), 2)      AS min_spend,
        ROUND(MAX(total_spend), 2)      AS max_spend,
        ROUND(PERCENTILE(total_spend, 0.5), 2)  AS median_spend,
        ROUND(PERCENTILE(total_spend, 0.75), 2) AS p75_spend,
        ROUND(PERCENTILE(total_spend, 0.90), 2) AS p90_spend

    FROM customer_totals
""")

print("CUSTOMER SPEND SUMMARY")
df_spend.show(truncate=False)

df_spend.repartition(2).write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.customer_spend_summary")

print(f"gold.customer_spend_summary written")

In [0]:
df_orders_per_customer = spark.sql("""
    WITH customer_orders AS (
        SELECT
            cu.customer_sk,
            cu.customer_id,
            cu.age_band,
            cu.gender,
            cu.loyalty_label,
            COUNT(f.order_id)                               AS total_orders,
            ROUND(SUM(f.revenue),  2)                       AS total_revenue,
            ROUND(SUM(f.revenue) / COUNT(f.order_id), 2)   AS aov

        FROM fact_sales    f
        JOIN dim_customers cu ON f.customer_sk = cu.customer_sk
        WHERE f.is_valid = 1
        GROUP BY
            cu.customer_sk, cu.customer_id,
            cu.age_band, cu.gender, cu.loyalty_label
    )
    SELECT
        *,
        CASE
            WHEN total_orders >= 10 THEN 'Frequent (10+)'
            WHEN total_orders >= 5  THEN 'Regular (5-9)'
            WHEN total_orders >= 2  THEN 'Occasional (2-4)'
            ELSE 'One-time buyer'
        END AS frequency_segment
    FROM customer_orders
    ORDER BY total_orders DESC
""")

print("ORDER FREQUENCY DISTRIBUTION")
df_orders_per_customer.groupBy("frequency_segment") \
    .count().orderBy("count", ascending=False).show()

row_count = df_orders_per_customer.count()
num_partitions = max(2, math.ceil(row_count / 10000))

df_orders_per_customer.repartition(num_partitions).write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.orders_per_customer")

print(f"gold.orders_per_customer written - {row_count} customers ({num_partitions} partitions)")

In [0]:
print("GOLD - Customer Analysis COMPLETE")
print("  gold.customer_revenue_ranking - Top customers")
print("  gold.revenue_by_age_band      - Revenue by age")
print("  gold.loyalty_vs_nonloyalty     - Loyalty impact")
print("  gold.customer_spend_summary   - Avg spend")
print("  gold.orders_per_customer      - Order frequency")
print()

for tbl in ["customer_revenue_ranking", "revenue_by_age_band", "loyalty_vs_nonloyalty", "customer_spend_summary", "orders_per_customer"]:
    cnt = spark.table(f"{GOLD_DB}.{tbl}").count()
    print(f"   gold.{tbl:<30} - {cnt:,} rows")